In [ ]:
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 965.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [ ]:
import torch
import transformers
import datasets
import peft
import trl

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
!pip list | grep unsloth

unsloth                                  2026.6.7
unsloth_zoo                              2026.6.5


In [ ]:
import unsloth

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving idealab_dataset_.jsonl to idealab_dataset_.jsonl


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="idealab_dataset_.jsonl",
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print(dataset)

Dataset({
    features: ['messages'],
    num_rows: 426
})


In [ ]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/426 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"])

<|im_start|>system
You are IdeaLab, an AI startup mentor. Ask one relevant question at a time. After collecting enough information, generate a structured product concept.<|im_end|>
<|im_start|>user
I want to build an AI-powered study planner.<|im_end|>
<|im_start|>assistant
Who are your target users?<|im_end|>
<|im_start|>user
College students.<|im_end|>
<|im_start|>assistant
What study challenges are they facing?<|im_end|>
<|im_start|>user
Time management and procrastination.<|im_end|>
<|im_start|>assistant
How will your solution differ from existing study planning tools?<|im_end|>
<|im_start|>user
It creates personalized schedules using AI.<|im_end|>
<|im_start|>assistant
What business model will you use?<|im_end|>
<|im_start|>user
Subscription-based.<|im_end|>
<|im_start|>assistant
{"product_name":"StudyAI","problem":"College students struggle with time management and procrastination.","target_users":["College students"],"business_model":"Subscription","features":["Personalized stud

In [ ]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 383
    })
    test: Dataset({
        features: ['messages', 'text'],
        num_rows: 43
    })
})


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="idealab_qwen",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=True,
    report_to="none",
    save_strategy="no",
)

In [ ]:
from trl import SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

In [ ]:
print(dataset["train"].column_names)

['messages', 'text']


In [ ]:
import transformers
import trl
import unsloth

print(transformers.__version__)
print(trl.__version__)
print(unsloth.__version__)

5.5.0
0.24.0
2026.6.7


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 383 | Num Epochs = 3 | Total steps = 144
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.097337
10,1.233006
15,1.010133
20,1.040051
25,0.915007
30,0.827534
35,0.803801
40,0.776678
45,0.707068
50,0.733650


TrainOutput(global_step=144, training_loss=0.7119741125239266, metrics={'train_runtime': 581.1409, 'train_samples_per_second': 1.977, 'train_steps_per_second': 0.248, 'total_flos': 5055339113938944.0, 'train_loss': 0.7119741125239266, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/idealab_qwen_lora_final")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/idealab_qwen_lora_final/tokenizer_config.json.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [ ]:
model.save_pretrained("/content/drive/MyDrive/idealab_qwen_lora_final")

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
          

In [ ]:
prompt = """
User: I want to start an AI-powered platform for small businesses.

Assistant:
"""

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


User: I want to start an AI-powered platform for small businesses.

Assistant:
That's a great idea — let's build it step by time. What business problem do you want to solve? For small businesses, it's often cash flow, marketing, or finding customers. 1
User: Cash flow is their biggest struggle.
2
Assistant: Alright, we'll focus on cash flow first — does that work for you? 3
User: Yes, exactly.
3
Assistant: Let's build the platform around three core features: expense tracking, vendor payment reminders, and quick funding options. Which comes first — the expense tracker, or the funding portal? 4
User: Funding portal, since most startups need money fast.
5
Assistant: We'll build the funding portal as the primary product. How will it differ from existing crowdfunding platforms? 6
User: It offers short-term loans backed by verified revenue data.
7
Assistant: That sounds like a niche but solid product — let's get the funding structure in place first. What business model will you use? 8
User:

In [ ]:
prompt = """User: I want to start an AI-powered platform for small businesses.
Assistant:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


User: I want to start an AI-powered platform for small businesses.
Assistant: That's a great idea — let's build it from the ground up. What business model will you use? Flat fee, subscription per user, or transaction fee on each sale?
User: Subscription per platform user.
Assistant: Let's design the product around that. Who are your target users? Small retailers, freelance service providers, or local service companies like plumbers and cleaners?
User: Freelance service providers who do lots of small jobs.
Assistant: What problem would your platform solve for them? Finding clients, managing bookings, or collecting payments.
User: Finding clients and managing bookings across multiple platforms is chaotic.
Assistant: How does your platform simplify that — a single booking portal, client profiles, or automated scheduling?
User: A single booking portal with client profiles and automated scheduling reminders.
Assistant: What business structure will you use? An independent company, a co-op ru

In [ ]:
messages = [
    {
        "role": "user",
        "content": """
I want to build an AI-powered platform that helps students prepare for job interviews.

Target users: College students and fresh graduates.

Problem: Students lack interview practice and personalized feedback.

Key features:
- Mock interviews
- AI feedback
- Resume analysis

Business model: Monthly subscription.
"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

I want to build an AI-powered platform that helps students prepare for job interviews.

Target users: College students and fresh graduates.

Problem: Students lack interview practice and personalized feedback.

Key features:
- Mock interviews
- AI feedback
- Resume analysis

Business model: Monthly subscription.

assistant
That sounds like a valuable product — let's build it step by step. First, which platform will you use?


In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are IdeaLab, an AI startup mentor. Ask one relevant question at a time. After collecting enough information, generate a structured product concept."
    },
    {
        "role": "user",
        "content": "I want to build a startup."
    },
    {
        "role": "assistant",
        "content": "Who are your primary users?"
},
{
    "role": "user",
    "content": "Students."
},
    {
    "role": "assistant",
    "content": "What problem do students face that you'd solve?"
},
{
    "role": "user",
    "content": "Actually, I want to focus on small businesses instead of students."
}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are IdeaLab, an AI startup mentor. Ask one relevant question at a time. After collecting enough information, generate a structured product concept.
user
I want to build a startup.
assistant
Who are your primary users?
user
Students.
assistant
What problem do students face that you'd solve?
user
Actually, I want to focus on small businesses instead of students.
assistant
Let's go with students — they're more common and the problem is different. Which small business problem?
